In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import StratifiedGroupKFold

from lifelines import (
    CoxPHFitter,
    KaplanMeierFitter
)

from sksurv.util import Surv
from sksurv.metrics import (
    concordance_index_censored,
    brier_score,
    integrated_brier_score,
    cumulative_dynamic_auc
)

from sksurv.linear_model import (
    CoxPHSurvivalAnalysis
)

from sksurv.ensemble import (
    RandomSurvivalForest,
    GradientBoostingSurvivalAnalysis
)

from sksurv.svm import (
    FastSurvivalSVM
)

import torch
import torchtuples as tt

from pycox.models import (
    CoxPH as DeepSurvCoxPH
)

In [ ]:
def survival_model_benchmark(
    df,
    duration_col="remainingDays",
    event_col="eventHasHappend",
    feature_columns=None,
    exclude_columns=None,
    test_size=0.20,
    random_state=42,
    cox_penalizer=0.1,
    rsf_estimators=300,
    gb_estimators=200,
    deepsurv_epochs=100,
    deepsurv_nodes=(64, 32),
    show_km=True,
    verbose=True
):
    """
    Benchmark several survival-analysis approaches on one common
    train/test split.

    Models
    ------
    1. Linear Regression
       Naive baseline; does NOT correctly handle censoring.

    2. Kaplan-Meier
       Population-level survival estimator, not individualized prediction.

    3. Cox Model
       scikit-survival CoxPHSurvivalAnalysis.

    4. CoxPH
       lifelines CoxPHFitter with optional penalization.

    5. Random Survival Forest

    6. Gradient Boosting Survival Analysis

    7. Survival Support Vector Machine

    8. DeepSurv
       Neural-network Cox proportional hazards model via PyCox.

    Returns
    -------
    dict
        {
            "results": comparison DataFrame,
            "models": fitted models,
            "preprocessing": fitted preprocessing objects,
            "train_index": ...,
            "test_index": ...,
            "km": KaplanMeierFitter,
            "feature_columns": ...
        }
    """

    # =========================================================
    # 1. PREPARE DATA
    # =========================================================

    data = df.copy()

    data[duration_col] = pd.to_numeric(
        data[duration_col],
        errors="coerce"
    )

    data[event_col] = (
        data[event_col]
        .astype(bool)
        .astype(int)
    )

    # Remove invalid outcome rows
    data = data[
        data[duration_col].notna()
        & (data[duration_col] > 0)
        & data[event_col].notna()
    ].copy()

    # =========================================================
    # 2. EXCLUDE LEAKAGE / NON-PREDICTOR VARIABLES
    # =========================================================

    default_exclusions = [
        "id",

        # Outcomes
        duration_col,
        event_col,

        # Future information
        "targetEndDate",
        "assignmentsAfterCut",
        "ausgesch-am",

        # Dates / observation markers
        "cutDate",
        "startofCaregiver",
        "endofCaregiver",
        "eingestellt-am"
    ]

    if exclude_columns is not None:
        default_exclusions += exclude_columns

    default_exclusions = list(
        set(default_exclusions)
    )

    # =========================================================
    # 3. SELECT FEATURES
    # =========================================================

    if feature_columns is None:

        feature_columns = (
            data
            .select_dtypes(
                include=["number", "bool"]
            )
            .columns
            .difference(default_exclusions)
            .tolist()
        )

    else:

        feature_columns = [
            col
            for col in feature_columns
            if col in data.columns
            and col not in default_exclusions
        ]

    X = data[feature_columns].copy()

    # Convert bool -> int
    for col in X.columns:

        if pd.api.types.is_bool_dtype(X[col]):
            X[col] = X[col].astype(int)

        else:
            X[col] = pd.to_numeric(
                X[col],
                errors="coerce"
            )

    # Remove completely missing columns
    X = X.loc[
        :,
        ~X.isna().all()
    ]

    # Remove constants
    variable_columns = [
        col
        for col in X.columns
        if X[col].nunique(dropna=True) > 1
    ]

    X = X[variable_columns]

    feature_columns = variable_columns

    durations = data[duration_col]
    events = data[event_col]

    # =========================================================
    # 4. TRAIN / TEST SPLIT
    # =========================================================

    train_idx, test_idx = train_test_split(
        np.arange(len(data)),
        test_size=test_size,
        random_state=random_state,
        stratify=events
    )

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    duration_train = durations.iloc[train_idx].to_numpy()
    duration_test = durations.iloc[test_idx].to_numpy()

    event_train = (
        events.iloc[train_idx]
        .astype(bool)
        .to_numpy()
    )

    event_test = (
        events.iloc[test_idx]
        .astype(bool)
        .to_numpy()
    )

    # =========================================================
    # 5. IMPUTATION
    # =========================================================

    imputer = SimpleImputer(
        strategy="median"
    )

    X_train_imputed = imputer.fit_transform(
        X_train
    )

    X_test_imputed = imputer.transform(
        X_test
    )

    # =========================================================
    # 6. STANDARDIZATION
    # =========================================================

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train_imputed
    )

    X_test_scaled = scaler.transform(
        X_test_imputed
    )

    # Structured survival outcomes
    y_train = Surv.from_arrays(
        event=event_train,
        time=duration_train
    )

    y_test = Surv.from_arrays(
        event=event_test,
        time=duration_test
    )

    # =========================================================
    # Helper: C-index
    # =========================================================

    def calculate_cindex(risk_score):

        return concordance_index_censored(
            event_test,
            duration_test,
            np.asarray(risk_score)
        )[0]

    results = []
    models = {}

    # =========================================================
    # 7. BASIC DATASET INFORMATION
    # =========================================================

    if verbose:

        print("=======================================")
        print("SURVIVAL BENCHMARK")
        print("=======================================")

        print(
            f"Observations: {len(data)}"
        )

        print(
            f"Features: {len(feature_columns)}"
        )

        print(
            f"Training observations: {len(train_idx)}"
        )

        print(
            f"Testing observations: {len(test_idx)}"
        )

        print(
            f"Events: {events.sum()}"
        )

        print(
            f"Censored: {(events == 0).sum()}"
        )

        print(
            f"Event rate: {events.mean():.2%}"
        )

    # =========================================================
    # 8. KAPLAN-MEIER
    # =========================================================

    kmf = KaplanMeierFitter()

    kmf.fit(
        duration_train,
        event_observed=event_train,
        label="Training population"
    )

    models["Kaplan-Meier"] = kmf

    results.append({
        "model": "Kaplan-Meier",
        "model_type": "Population estimator",
        "c_index": np.nan,
        "notes":
            "Population survival baseline; "
            "no individualized risk score."
    })

    if show_km:

        fig, ax = plt.subplots(
            figsize=(8, 5)
        )

        kmf.plot_survival_function(
            ax=ax,
            ci_show=True
        )

        ax.set_title(
            "Kaplan-Meier Survival Estimate"
        )

        ax.set_xlabel(
            "Remaining Days"
        )

        ax.set_ylabel(
            "Survival Probability"
        )

        ax.set_ylim(
            0,
            1.05
        )

        plt.tight_layout()
        plt.show()

    # =========================================================
    # 9. LINEAR REGRESSION
    # =========================================================
    #
    # Naive baseline:
    # train only on observed events.
    #
    # This deliberately ignores censored durations.
    # =========================================================

    event_rows_train = event_train == True

    linear = LinearRegression()

    linear.fit(
        X_train_scaled[event_rows_train],
        duration_train[event_rows_train]
    )

    predicted_duration = linear.predict(
        X_test_scaled
    )

    # Short predicted survival = high risk
    linear_risk = -predicted_duration

    linear_cindex = calculate_cindex(
        linear_risk
    )

    models["Linear Regression"] = linear

    results.append({
        "model": "Linear Regression",
        "model_type": "Naive regression baseline",
        "c_index": linear_cindex,
        "notes":
            "Does not properly model censoring."
    })

    # =========================================================
    # 10. SCIKIT-SURVIVAL COX MODEL
    # =========================================================

    try:

        cox_model = CoxPHSurvivalAnalysis(
            alpha=0.01
        )

        cox_model.fit(
            X_train_scaled,
            y_train
        )

        risk_cox = cox_model.predict(
            X_test_scaled
        )

        cox_cindex = calculate_cindex(
            risk_cox
        )

        models["Cox Model"] = cox_model

        results.append({
            "model": "Cox Model",
            "model_type": "Cox proportional hazards",
            "c_index": cox_cindex,
            "notes":
                "scikit-survival implementation."
        })

    except Exception as e:

        results.append({
            "model": "Cox Model",
            "model_type": "Cox proportional hazards",
            "c_index": np.nan,
            "notes": f"Failed: {e}"
        })

    # =========================================================
    # 11. LIFELINES COXPH
    # =========================================================

    try:

        train_cox = pd.DataFrame(
            X_train_scaled,
            columns=feature_columns
        )

        train_cox[duration_col] = (
            duration_train
        )

        train_cox[event_col] = (
            event_train.astype(int)
        )

        test_cox = pd.DataFrame(
            X_test_scaled,
            columns=feature_columns
        )

        cph = CoxPHFitter(
            penalizer=cox_penalizer
        )

        cph.fit(
            train_cox,
            duration_col=duration_col,
            event_col=event_col
        )

        coxph_risk = (
            cph.predict_partial_hazard(
                test_cox
            )
            .to_numpy()
            .ravel()
        )

        coxph_cindex = calculate_cindex(
            coxph_risk
        )

        models["CoxPH"] = cph

        results.append({
            "model": "CoxPH",
            "model_type": "Penalized Cox PH",
            "c_index": coxph_cindex,
            "notes":
                f"lifelines, penalizer={cox_penalizer}."
        })

    except Exception as e:

        results.append({
            "model": "CoxPH",
            "model_type": "Penalized Cox PH",
            "c_index": np.nan,
            "notes": f"Failed: {e}"
        })

    # =========================================================
    # 12. RANDOM SURVIVAL FOREST
    # =========================================================

    try:

        rsf = RandomSurvivalForest(
            n_estimators=rsf_estimators,
            min_samples_split=10,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=random_state
        )

        # Scaling isn't required by trees, so use imputed values
        rsf.fit(
            X_train_imputed,
            y_train
        )

        rsf_risk = rsf.predict(
            X_test_imputed
        )

        rsf_cindex = calculate_cindex(
            rsf_risk
        )

        models[
            "Random Survival Forest"
        ] = rsf

        results.append({
            "model":
                "Random Survival Forest",

            "model_type":
                "Tree ensemble",

            "c_index":
                rsf_cindex,

            "notes":
                f"{rsf_estimators} trees."
        })

    except Exception as e:

        results.append({
            "model":
                "Random Survival Forest",

            "model_type":
                "Tree ensemble",

            "c_index":
                np.nan,

            "notes":
                f"Failed: {e}"
        })

    # =========================================================
    # 13. GRADIENT BOOSTING SURVIVAL
    # =========================================================

    try:

        gb = GradientBoostingSurvivalAnalysis(
            loss="coxph",
            n_estimators=gb_estimators,
            learning_rate=0.05,
            max_depth=3,
            random_state=random_state
        )

        gb.fit(
            X_train_imputed,
            y_train
        )

        gb_risk = gb.predict(
            X_test_imputed
        )

        gb_cindex = calculate_cindex(
            gb_risk
        )

        models[
            "Gradient Boosting Survival"
        ] = gb

        results.append({
            "model":
                "Gradient Boosting Survival",

            "model_type":
                "Boosted survival trees",

            "c_index":
                gb_cindex,

            "notes":
                f"{gb_estimators} estimators."
        })

    except Exception as e:

        results.append({
            "model":
                "Gradient Boosting Survival",

            "model_type":
                "Boosted survival trees",

            "c_index":
                np.nan,

            "notes":
                f"Failed: {e}"
        })

    # =========================================================
    # 14. SURVIVAL SUPPORT VECTOR MACHINE
    # =========================================================

    try:

        survival_svm = FastSurvivalSVM(
            alpha=1.0,
            rank_ratio=1.0,
            max_iter=1000,
            random_state=random_state
        )

        survival_svm.fit(
            X_train_scaled,
            y_train
        )

        svm_risk = survival_svm.predict(
            X_test_scaled
        )

        svm_cindex = calculate_cindex(
            svm_risk
        )

        models[
            "Survival SVM"
        ] = survival_svm

        results.append({
            "model":
                "Survival SVM",

            "model_type":
                "Ranking survival model",

            "c_index":
                svm_cindex,

            "notes":
                "FastSurvivalSVM."
        })

    except Exception as e:

        results.append({
            "model":
                "Survival SVM",

            "model_type":
                "Ranking survival model",

            "c_index":
                np.nan,

            "notes":
                f"Failed: {e}"
        })

    # =========================================================
    # 15. DEEPSURV
    # =========================================================

    try:

        np.random.seed(
            random_state
        )

        torch.manual_seed(
            random_state
        )

        x_train_deep = (
            X_train_scaled
            .astype("float32")
        )

        x_test_deep = (
            X_test_scaled
            .astype("float32")
        )

        duration_train_deep = (
            duration_train
            .astype("float32")
        )

        event_train_deep = (
            event_train
            .astype("float32")
        )

        # ---------------------------------------------
        # Neural network
        # ---------------------------------------------

        in_features = (
            x_train_deep.shape[1]
        )

        net = tt.practical.MLPVanilla(
            in_features,
            list(deepsurv_nodes),
            1,
            batch_norm=True,
            dropout=0.1,
            output_bias=False
        )

        deepsurv = DeepSurvCoxPH(
            net,
            tt.optim.Adam
        )

        deepsurv.optimizer.set_lr(
            0.001
        )

        # Validation split from training data
        (
            x_deep_train,
            x_deep_val,
            duration_deep_train,
            duration_deep_val,
            event_deep_train,
            event_deep_val
        ) = train_test_split(
            x_train_deep,
            duration_train_deep,
            event_train_deep,
            test_size=0.20,
            random_state=random_state,
            stratify=event_train_deep
        )

        callbacks = [
            tt.callbacks.EarlyStopping(
                patience=10
            )
        ]

        deepsurv.fit(
            x_deep_train,
            (
                duration_deep_train,
                event_deep_train
            ),
            batch_size=256,
            epochs=deepsurv_epochs,
            callbacks=callbacks,
            val_data=(
                x_deep_val,
                (
                    duration_deep_val,
                    event_deep_val
                )
            ),
            verbose=False
        )

        # DeepSurv returns log-risk
        deep_risk = (
            deepsurv.predict(
                x_test_deep
            )
            .reshape(-1)
        )

        deep_cindex = calculate_cindex(
            deep_risk
        )

        models[
            "DeepSurv"
        ] = deepsurv

        results.append({
            "model":
                "DeepSurv",

            "model_type":
                "Neural Cox-PH",

            "c_index":
                deep_cindex,

            "notes":
                f"Hidden layers: {deepsurv_nodes}."
        })

    except Exception as e:

        results.append({
            "model":
                "DeepSurv",

            "model_type":
                "Neural Cox-PH",

            "c_index":
                np.nan,

            "notes":
                f"Failed: {e}"
        })

    # =========================================================
    # 16. RESULTS TABLE
    # =========================================================

    results_df = pd.DataFrame(
        results
    )

    results_df = (
        results_df
        .sort_values(
            "c_index",
            ascending=False,
            na_position="last"
        )
        .reset_index(drop=True)
    )

    if verbose:

        print(
            "\n======================================="
        )

        print(
            "MODEL COMPARISON"
        )

        print(
            "======================================="
        )

        print(
            results_df[
                [
                    "model",
                    "c_index",
                    "notes"
                ]
            ]
        )

    return {
        "results":
            results_df,

        "models":
            models,

        "kaplan_meier":
            kmf,

        "feature_columns":
            feature_columns,

        "imputer":
            imputer,

        "scaler":
            scaler,

        "train_index":
            data.index[train_idx],

        "test_index":
            data.index[test_idx],

        "X_train":
            X_train,

        "X_test":
            X_test,

        "y_train":
            y_train,

        "y_test":
            y_test
    }

In [ ]:
temporal_df = pd.read_csv("../combination/ratio_modelData.csv", sep=";")

In [ ]:
def evaluate_survival_predictions(
    model_name,
    event_train,
    duration_train,
    event_test,
    duration_test,
    risk_scores,
    survival_probabilities=None,
    survival_times=None,
    horizons=(7, 30, 180, 365)
):
    """
    Calculate survival-model evaluation metrics.

    Parameters
    ----------
    risk_scores : array
        Higher value = greater risk.

    survival_probabilities : ndarray or None
        Shape:
            (n_test, n_times)

        Survival probability S(t) for every test observation
        at every value in survival_times.

    survival_times : ndarray or None
        Times corresponding to survival_probabilities.

    Returns
    -------
    dict
    """

    result = {
        "model": model_name
    }

    # ---------------------------------------------------------
    # Structured outcomes
    # ---------------------------------------------------------

    y_train = Surv.from_arrays(
        event=event_train.astype(bool),
        time=duration_train
    )

    y_test = Surv.from_arrays(
        event=event_test.astype(bool),
        time=duration_test
    )

    # =========================================================
    # 1. C-INDEX
    # =========================================================

    try:

        result["c_index"] = (
            concordance_index_censored(
                event_test.astype(bool),
                duration_test,
                np.asarray(risk_scores)
            )[0]
        )

    except Exception:

        result["c_index"] = np.nan

    # Defaults for probability-dependent metrics
    result["brier_365"] = np.nan
    result["integrated_brier_score"] = np.nan

    for horizon in horizons:
        result[f"auc_{horizon}"] = np.nan
        result[f"calibration_{horizon}"] = np.nan

    result["mae_events"] = np.nan
    result["rmse_events"] = np.nan

    # Stop here if no survival function exists
    if (
        survival_probabilities is None
        or survival_times is None
    ):
        return result

    survival_probabilities = np.asarray(
        survival_probabilities
    )

    survival_times = np.asarray(
        survival_times
    )

    # =========================================================
    # Helper: interpolate S(t)
    # =========================================================

    def survival_at_time(t):

        values = []

        for curve in survival_probabilities:

            values.append(
                np.interp(
                    t,
                    survival_times,
                    curve,
                    left=1.0,
                    right=curve[-1]
                )
            )

        return np.asarray(values)

    # =========================================================
    # 2. BRIER SCORE
    # =========================================================

    # Example single-time Brier score at 365 days
    brier_time = 365

    try:

        if (
            brier_time < duration_train.max()
            and brier_time < duration_test.max()
        ):

            survival_365 = survival_at_time(
                brier_time
            )

            _, bs = brier_score(
                y_train,
                y_test,
                survival_365.reshape(-1, 1),
                np.array([brier_time])
            )

            result["brier_365"] = (
                bs[0]
            )

    except Exception:
        pass

    # =========================================================
    # 3. INTEGRATED BRIER SCORE
    # =========================================================

    try:

        lower = max(
            1,
            np.percentile(
                duration_test,
                5
            )
        )

        upper = min(
            np.percentile(
                duration_test,
                90
            ),
            duration_train.max() - 1
        )

        ibs_times = np.linspace(
            lower,
            upper,
            100
        )

        survival_matrix = np.column_stack([
            survival_at_time(t)
            for t in ibs_times
        ])

        result[
            "integrated_brier_score"
        ] = integrated_brier_score(
            y_train,
            y_test,
            survival_matrix,
            ibs_times
        )

    except Exception:
        pass

    # =========================================================
    # 4. TIME-DEPENDENT AUC
    # =========================================================

    for horizon in horizons:

        try:

            if (
                horizon < duration_train.max()
                and horizon < duration_test.max()
            ):

                auc, _ = (
                    cumulative_dynamic_auc(
                        y_train,
                        y_test,
                        np.asarray(
                            risk_scores
                        ),
                        np.array([
                            horizon
                        ])
                    )
                )

                result[
                    f"auc_{horizon}"
                ] = auc[0]

        except Exception:
            pass

    # =========================================================
    # 5. EXPECTED SURVIVAL TIME
    # =========================================================
    #
    # E[T] ≈ integral S(t) dt
    #
    # This gives us an approximate predicted remaining
    # survival duration.
    # =========================================================

    try:

        predicted_time = np.trapz(
            survival_probabilities,
            survival_times,
            axis=1
        )

        # -----------------------------------------------------
        # MAE / RMSE
        #
        # Only evaluate uncensored observations.
        # -----------------------------------------------------

        observed = (
            event_test == True
        )

        if observed.sum() > 0:

            result["mae_events"] = (
                mean_absolute_error(
                    duration_test[
                        observed
                    ],
                    predicted_time[
                        observed
                    ]
                )
            )

            result["rmse_events"] = (
                np.sqrt(
                    mean_squared_error(
                        duration_test[
                            observed
                        ],
                        predicted_time[
                            observed
                        ]
                    )
                )
            )

    except Exception:
        pass

    # =========================================================
    # 6. CALIBRATION
    # =========================================================
    #
    # Simple calibration error:
    #
    # predicted survival probability
    # versus
    # Kaplan-Meier observed survival.
    #
    # Lower = better.
    # =========================================================

    for horizon in horizons:

        try:

            predicted_survival = (
                survival_at_time(
                    horizon
                )
            )

            mean_predicted_survival = (
                predicted_survival.mean()
            )

            km = KaplanMeierFitter()

            km.fit(
                duration_test,
                event_observed=event_test
            )

            observed_survival = (
                km.predict(
                    horizon
                )
            )

            calibration_error = abs(
                mean_predicted_survival
                - observed_survival
            )

            result[
                f"calibration_{horizon}"
            ] = calibration_error

        except Exception:
            pass

    return result

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import StratifiedGroupKFold

from lifelines import CoxPHFitter, KaplanMeierFitter

from sksurv.util import Surv
from sksurv.metrics import (
    concordance_index_censored,
    brier_score,
    integrated_brier_score,
    cumulative_dynamic_auc
)

from sksurv.linear_model import CoxPHSurvivalAnalysis

from sksurv.ensemble import (
    RandomSurvivalForest,
    GradientBoostingSurvivalAnalysis
)

from sksurv.svm import FastSurvivalSVM

import torch
import torchtuples as tt
from pycox.models import CoxPH as DeepSurvCoxPH


def cross_validate_survival_models(
    df,
    duration_col="remainingDays",
    event_col="eventHasHappend",
    group_col="id",
    feature_columns=None,
    exclude_columns=None,
    n_splits=5,
    random_state=42,
    horizons=(7, 30, 180, 365),
    calibration_bins=10,
    restricted_time_horizon=365,
    cox_penalizer=0.1,
    rsf_estimators=300,
    gb_estimators=200,
    deepsurv_epochs=100,
    deepsurv_nodes=(64, 32),
    verbose=True
):

    # =========================================================
    # PREPARE DATA
    # =========================================================

    data = df.copy()

    data[duration_col] = pd.to_numeric(
        data[duration_col],
        errors="coerce"
    )

    data[event_col] = (
        data[event_col]
        .astype(bool)
        .astype(int)
    )

    data = data[
        data[duration_col].notna()
        & (data[duration_col] > 0)
        & data[event_col].notna()
    ].copy()


    # =========================================================
    # EXCLUSIONS
    # =========================================================

    default_exclusions = [
        group_col,
        duration_col,
        event_col,

        "targetEndDate",
        "assignmentsAfterCut",
        "ausgesch-am",

        "cutDate",
        "startofCaregiver",
        "endofCaregiver",
        "eingestellt-am"
    ]

    if exclude_columns is not None:
        default_exclusions += exclude_columns

    default_exclusions = list(
        set(default_exclusions)
    )


    # =========================================================
    # FEATURE SELECTION
    # =========================================================

    if feature_columns is None:

        feature_columns = (
            data
            .select_dtypes(
                include=["number", "bool"]
            )
            .columns
            .difference(default_exclusions)
            .tolist()
        )

    else:

        feature_columns = [
            col
            for col in feature_columns
            if col in data.columns
            and col not in default_exclusions
        ]


    X = data[
        feature_columns
    ].copy()


    # =========================================================
    # CONVERT FEATURES
    # =========================================================

    for col in X.columns:

        if pd.api.types.is_bool_dtype(
            X[col]
        ):

            X[col] = (
                X[col]
                .astype(int)
            )

        else:

            X[col] = pd.to_numeric(
                X[col],
                errors="coerce"
            )


    # Remove all-missing columns
    X = X.loc[
        :,
        ~X.isna().all()
    ]


    # Remove globally constant columns
    variable_columns = [
        col
        for col in X.columns
        if X[col].nunique(
            dropna=True
        ) > 1
    ]

    X = X[
        variable_columns
    ]

    feature_columns = variable_columns


    # =========================================================
    # TARGET ARRAYS
    # =========================================================

    durations = (
        data[duration_col]
        .to_numpy()
        .astype(float)
    )

    events = (
        data[event_col]
        .astype(bool)
        .to_numpy()
    )

    groups = (
        data[group_col]
        .to_numpy()
    )


    # =========================================================
    # CROSS VALIDATION
    # =========================================================

    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    fold_results = []
    calibration_results = []
    prediction_results = []


    # =========================================================
    # HELPER:
    # MEDIAN SURVIVAL TIME
    # =========================================================

    def calculate_median_survival_times(
        survival_probabilities,
        survival_times,
        max_time
    ):
        """
        Median survival time:
            first t where S(t) <= 0.5

        Uses linear interpolation around the 0.5 crossing.

        If the curve never reaches 0.5 before max_time,
        prediction is capped at max_time.

        Returns
        -------
        median_times : np.ndarray
        median_reached : np.ndarray of bool
        """

        survival_probabilities = np.asarray(
            survival_probabilities
        )

        survival_times = np.asarray(
            survival_times,
            dtype=float
        )

        median_times = []
        median_reached = []


        for curve in survival_probabilities:

            # Only consider points up to max_time
            valid = (
                survival_times <= max_time
            )

            times = survival_times[
                valid
            ]

            probs = curve[
                valid
            ]


            if len(times) == 0:

                median_times.append(
                    max_time
                )

                median_reached.append(
                    False
                )

                continue


            crossing = np.where(
                probs <= 0.5
            )[0]


            # ---------------------------------------------
            # Never reaches 50%
            # ---------------------------------------------

            if len(crossing) == 0:

                median_times.append(
                    max_time
                )

                median_reached.append(
                    False
                )

                continue


            idx = crossing[0]


            # ---------------------------------------------
            # Already <= .5 at first time point
            # ---------------------------------------------

            if idx == 0:

                median = times[0]


            # ---------------------------------------------
            # Interpolate crossing
            # ---------------------------------------------

            else:

                t1 = times[
                    idx - 1
                ]

                t2 = times[
                    idx
                ]

                s1 = probs[
                    idx - 1
                ]

                s2 = probs[
                    idx
                ]


                if s1 == s2:

                    median = t2

                else:

                    median = (
                        t1
                        +
                        (0.5 - s1)
                        * (t2 - t1)
                        / (s2 - s1)
                    )


            median_times.append(
                min(
                    float(median),
                    max_time
                )
            )

            median_reached.append(
                True
            )


        return (
            np.asarray(
                median_times
            ),
            np.asarray(
                median_reached
            )
        )


    # =========================================================
    # MODEL EVALUATION
    # =========================================================

    def evaluate_model(
        model_name,
        event_train,
        duration_train,
        event_test,
        duration_test,
        risk_scores,
        survival_probabilities=None,
        survival_times=None
    ):

        result = {
            "model": model_name
        }

        calibration_data = []

        predicted_rmst = None
        predicted_median = None
        median_reached = None


        # -----------------------------------------------------
        # Survival structures
        # -----------------------------------------------------

        y_train_eval = Surv.from_arrays(
            event=event_train.astype(bool),
            time=duration_train
        )

        y_test_eval = Surv.from_arrays(
            event=event_test.astype(bool),
            time=duration_test
        )


        # =====================================================
        # C-INDEX
        # =====================================================

        try:

            result["c_index"] = (
                concordance_index_censored(
                    event_test.astype(bool),
                    duration_test,
                    np.asarray(
                        risk_scores
                    )
                )[0]
            )

        except Exception:

            result["c_index"] = np.nan


        # =====================================================
        # DEFAULT VALUES
        # =====================================================

        for horizon in horizons:

            result[
                f"brier_{horizon}"
            ] = np.nan

            result[
                f"auc_{horizon}"
            ] = np.nan

            result[
                f"calibration_{horizon}"
            ] = np.nan


        result[
            "integrated_brier_score"
        ] = np.nan

        result[
            f"restricted_mae_{restricted_time_horizon}"
        ] = np.nan

        result[
            f"restricted_rmse_{restricted_time_horizon}"
        ] = np.nan


        # =====================================================
        # TIME-DEPENDENT AUC
        # =====================================================

        for horizon in horizons:

            try:

                if (
                    horizon < duration_train.max()
                    and
                    horizon < duration_test.max()
                ):

                    auc, _ = (
                        cumulative_dynamic_auc(
                            y_train_eval,
                            y_test_eval,
                            np.asarray(
                                risk_scores
                            ),
                            np.array([
                                horizon
                            ])
                        )
                    )

                    result[
                        f"auc_{horizon}"
                    ] = auc[0]

            except Exception:
                pass


        # =====================================================
        # NO SURVIVAL CURVES
        # =====================================================

        if (
            survival_probabilities is None
            or survival_times is None
        ):

            return (
                result,
                calibration_data,
                predicted_rmst,
                predicted_median,
                median_reached
            )


        survival_probabilities = (
            np.asarray(
                survival_probabilities
            )
        )

        survival_times = (
            np.asarray(
                survival_times,
                dtype=float
            )
        )


        # =====================================================
        # S(t)
        # =====================================================

        def survival_at_time(t):

            return np.asarray([
                np.interp(
                    t,
                    survival_times,
                    curve,
                    left=1.0,
                    right=curve[-1]
                )
                for curve
                in survival_probabilities
            ])


        # =====================================================
        # MEDIAN SURVIVAL TIME
        # =====================================================

        (
            predicted_median,
            median_reached
        ) = calculate_median_survival_times(
            survival_probabilities,
            survival_times,
            restricted_time_horizon
        )


        # =====================================================
        # BRIER SCORE
        # =====================================================

        for horizon in horizons:

            try:

                if (
                    horizon < duration_train.max()
                    and
                    horizon < duration_test.max()
                ):

                    survival_prediction = (
                        survival_at_time(
                            horizon
                        )
                    )

                    _, bs = brier_score(
                        y_train_eval,
                        y_test_eval,

                        survival_prediction.reshape(
                            -1,
                            1
                        ),

                        np.array([
                            horizon
                        ])
                    )

                    result[
                        f"brier_{horizon}"
                    ] = bs[0]

            except Exception:
                pass


        # =====================================================
        # INTEGRATED BRIER SCORE
        # =====================================================

        try:

            lower_time = max(
                1,
                np.percentile(
                    duration_test,
                    5
                )
            )

            upper_time = min(
                np.percentile(
                    duration_test,
                    90
                ),
                duration_train.max() - 1,
                duration_test.max() - 1
            )


            if upper_time > lower_time:

                ibs_times = np.linspace(
                    lower_time,
                    upper_time,
                    100
                )

                survival_matrix = (
                    np.column_stack([
                        survival_at_time(
                            t
                        )
                        for t
                        in ibs_times
                    ])
                )

                result[
                    "integrated_brier_score"
                ] = (
                    integrated_brier_score(
                        y_train_eval,
                        y_test_eval,
                        survival_matrix,
                        ibs_times
                    )
                )

        except Exception:
            pass


        # =====================================================
        # RESTRICTED MEAN SURVIVAL TIME
        # =====================================================

        try:

            tau = (
                restricted_time_horizon
            )

            integration_times = np.linspace(
                0,
                tau,
                200
            )

            survival_matrix_tau = (
                np.column_stack([
                    survival_at_time(
                        t
                    )
                    for t
                    in integration_times
                ])
            )

            predicted_rmst = (
                np.trapezoid(
                    survival_matrix_tau,
                    integration_times,
                    axis=1
                )
            )


            observed_restricted_time = (
                np.minimum(
                    duration_test,
                    tau
                )
            )


            # Event observed before tau OR
            # survived at least to tau
            evaluable = (
                (
                    event_test
                    &
                    (
                        duration_test
                        <= tau
                    )
                )
                |
                (
                    duration_test
                    >= tau
                )
            )


            if evaluable.sum() > 0:

                result[
                    f"restricted_mae_{tau}"
                ] = (
                    mean_absolute_error(
                        observed_restricted_time[
                            evaluable
                        ],
                        predicted_rmst[
                            evaluable
                        ]
                    )
                )

                result[
                    f"restricted_rmse_{tau}"
                ] = (
                    np.sqrt(
                        mean_squared_error(
                            observed_restricted_time[
                                evaluable
                            ],
                            predicted_rmst[
                                evaluable
                            ]
                        )
                    )
                )

        except Exception:

            predicted_rmst = None


        # =====================================================
        # CALIBRATION
        # =====================================================

        for horizon in horizons:

            try:

                predicted_survival = (
                    survival_at_time(
                        horizon
                    )
                )

                calibration_df = (
                    pd.DataFrame({
                        "predicted_survival":
                            predicted_survival,

                        "duration":
                            duration_test,

                        "event":
                            event_test
                    })
                )


                calibration_df[
                    "bin"
                ] = pd.qcut(
                    calibration_df[
                        "predicted_survival"
                    ],
                    q=calibration_bins,
                    duplicates="drop"
                )


                group_errors = []


                for (
                    bin_id,
                    group
                ) in calibration_df.groupby(
                    "bin",
                    observed=False
                ):

                    if len(group) < 2:
                        continue


                    mean_predicted = (
                        group[
                            "predicted_survival"
                        ].mean()
                    )


                    km = (
                        KaplanMeierFitter()
                    )

                    km.fit(
                        group[
                            "duration"
                        ],

                        event_observed=
                            group[
                                "event"
                            ]
                    )


                    observed_survival = float(
                        km.predict(
                            horizon
                        )
                    )


                    error = abs(
                        mean_predicted
                        -
                        observed_survival
                    )


                    group_errors.append(
                        error
                    )


                    calibration_data.append({
                        "model":
                            model_name,

                        "horizon":
                            horizon,

                        "bin":
                            str(bin_id),

                        "n":
                            len(group),

                        "predicted_survival":
                            mean_predicted,

                        "observed_survival":
                            observed_survival,

                        "absolute_error":
                            error
                    })


                if len(group_errors) > 0:

                    result[
                        f"calibration_{horizon}"
                    ] = np.mean(
                        group_errors
                    )

            except Exception:
                pass


        return (
            result,
            calibration_data,
            predicted_rmst,
            predicted_median,
            median_reached
        )


    # =========================================================
    # CV LOOP
    # =========================================================

    for fold, (
        train_idx,
        test_idx
    ) in enumerate(
        cv.split(
            X,
            events,
            groups
        ),
        start=1
    ):


        if verbose:

            print(
                f"\n============ "
                f"Fold {fold}/{n_splits} "
                f"============"
            )


        # -----------------------------------------------------
        # Split
        # -----------------------------------------------------

        X_train = (
            X.iloc[
                train_idx
            ].copy()
        )

        X_test = (
            X.iloc[
                test_idx
            ].copy()
        )


        duration_train = (
            durations[
                train_idx
            ]
        )

        duration_test = (
            durations[
                test_idx
            ]
        )


        event_train = (
            events[
                train_idx
            ]
        )

        event_test = (
            events[
                test_idx
            ]
        )


        ids_test = (
            groups[
                test_idx
            ]
        )


        # -----------------------------------------------------
        # Imputation
        # -----------------------------------------------------

        imputer = SimpleImputer(
            strategy="median"
        )

        X_train_imputed = (
            imputer.fit_transform(
                X_train
            )
        )

        X_test_imputed = (
            imputer.transform(
                X_test
            )
        )


        # -----------------------------------------------------
        # Scaling
        # -----------------------------------------------------

        scaler = StandardScaler()

        X_train_scaled = (
            scaler.fit_transform(
                X_train_imputed
            )
        )

        X_test_scaled = (
            scaler.transform(
                X_test_imputed
            )
        )


        # -----------------------------------------------------
        # Survival target
        # -----------------------------------------------------

        y_train = Surv.from_arrays(
            event=event_train,
            time=duration_train
        )


        # =====================================================
        # STORE OOF PREDICTIONS
        # =====================================================

        def store_predictions(
            model_name,
            risk,
            predicted_rmst=None,
            predicted_median=None,
            median_reached=None
        ):

            for i in range(
                len(test_idx)
            ):

                prediction_results.append({

                    group_col:
                        ids_test[i],

                    "fold":
                        fold,

                    "model":
                        model_name,

                    duration_col:
                        duration_test[i],

                    event_col:
                        bool(
                            event_test[i]
                        ),

                    # Restricted observed duration
                    "observed_restricted_time":
                        min(
                            duration_test[i],
                            restricted_time_horizon
                        ),

                    "risk_score":
                        risk[i]
                        if risk is not None
                        else np.nan,

                    "predicted_rmst":
                        predicted_rmst[i]
                        if predicted_rmst is not None
                        else np.nan,

                    "predicted_median_time":
                        predicted_median[i]
                        if predicted_median is not None
                        else np.nan,

                    "median_reached":
                        bool(
                            median_reached[i]
                        )
                        if median_reached is not None
                        else np.nan
                })


        # =====================================================
        # 1. LINEAR REGRESSION
        # =====================================================

        try:

            observed_train = (
                event_train
                == True
            )

            linear = (
                LinearRegression()
            )

            linear.fit(
                X_train_scaled[
                    observed_train
                ],
                duration_train[
                    observed_train
                ]
            )


            predicted_linear = (
                linear.predict(
                    X_test_scaled
                )
            )


            predicted_linear = (
                np.clip(
                    predicted_linear,
                    0,
                    restricted_time_horizon
                )
            )


            risk = (
                -predicted_linear
            )


            (
                metrics,
                calibration,
                _,
                _,
                _
            ) = evaluate_model(
                "Linear Regression",
                event_train,
                duration_train,
                event_test,
                duration_test,
                risk_scores=risk
            )


            metrics[
                "fold"
            ] = fold


            # Linear regression has a direct time prediction,
            # so use it as both comparison columns.
            store_predictions(
                "Linear Regression",
                risk=risk,
                predicted_rmst=
                    predicted_linear,
                predicted_median=
                    predicted_linear,
                median_reached=
                    np.ones(
                        len(predicted_linear),
                        dtype=bool
                    )
            )


            fold_results.append(
                metrics
            )


        except Exception as e:

            fold_results.append({
                "fold":
                    fold,
                "model":
                    "Linear Regression",
                "error":
                    str(e)
            })


        # =====================================================
        # 2. COX MODEL
        # =====================================================

        try:

            cox = (
                CoxPHSurvivalAnalysis(
                    alpha=0.01
                )
            )


            cox.fit(
                X_train_scaled,
                y_train
            )


            risk = (
                cox.predict(
                    X_test_scaled
                )
            )


            survival_functions = (
                cox
                .predict_survival_function(
                    X_test_scaled
                )
            )


            survival_times = (
                np.unique(
                    duration_train[
                        event_train
                    ]
                )
            )


            survival_probabilities = np.array([
                fn(
                    survival_times
                )
                for fn
                in survival_functions
            ])


            (
                metrics,
                calibration,
                predicted_rmst,
                predicted_median,
                median_reached
            ) = evaluate_model(
                "Cox Model",
                event_train,
                duration_train,
                event_test,
                duration_test,
                risk,
                survival_probabilities,
                survival_times
            )


            metrics[
                "fold"
            ] = fold


            for row in calibration:

                row[
                    "fold"
                ] = fold


            calibration_results.extend(
                calibration
            )


            fold_results.append(
                metrics
            )


            store_predictions(
                "Cox Model",
                risk,
                predicted_rmst,
                predicted_median,
                median_reached
            )


        except Exception as e:

            fold_results.append({
                "fold":
                    fold,
                "model":
                    "Cox Model",
                "error":
                    str(e)
            })


        # =====================================================
        # 3. COXPH
        # =====================================================

        try:

            train_cox = (
                pd.DataFrame(
                    X_train_scaled,
                    columns=
                        feature_columns
                )
            )

            train_cox[
                duration_col
            ] = duration_train

            train_cox[
                event_col
            ] = (
                event_train
                .astype(int)
            )


            test_cox = (
                pd.DataFrame(
                    X_test_scaled,
                    columns=
                        feature_columns
                )
            )


            cph = (
                CoxPHFitter(
                    penalizer=
                        cox_penalizer
                )
            )


            cph.fit(
                train_cox,
                duration_col=
                    duration_col,
                event_col=
                    event_col
            )


            risk = (
                cph
                .predict_partial_hazard(
                    test_cox
                )
                .to_numpy()
                .ravel()
            )


            survival_df = (
                cph
                .predict_survival_function(
                    test_cox
                )
            )


            survival_times = (
                survival_df
                .index
                .to_numpy(
                    dtype=float
                )
            )


            survival_probabilities = (
                survival_df
                .T
                .to_numpy()
            )


            (
                metrics,
                calibration,
                predicted_rmst,
                predicted_median,
                median_reached
            ) = evaluate_model(
                "CoxPH",
                event_train,
                duration_train,
                event_test,
                duration_test,
                risk,
                survival_probabilities,
                survival_times
            )


            metrics[
                "fold"
            ] = fold


            for row in calibration:

                row[
                    "fold"
                ] = fold


            calibration_results.extend(
                calibration
            )


            fold_results.append(
                metrics
            )


            store_predictions(
                "CoxPH",
                risk,
                predicted_rmst,
                predicted_median,
                median_reached
            )


        except Exception as e:

            fold_results.append({
                "fold":
                    fold,
                "model":
                    "CoxPH",
                "error":
                    str(e)
            })


        # =====================================================
        # 4. RANDOM SURVIVAL FOREST
        # =====================================================

        try:

            rsf = (
                RandomSurvivalForest(
                    n_estimators=
                        rsf_estimators,

                    min_samples_split=
                        10,

                    min_samples_leaf=
                        5,

                    max_features=
                        "sqrt",

                    n_jobs=
                        -1,

                    random_state=
                        random_state
                )
            )


            rsf.fit(
                X_train_imputed,
                y_train
            )


            risk = (
                rsf.predict(
                    X_test_imputed
                )
            )


            survival_functions = (
                rsf
                .predict_survival_function(
                    X_test_imputed
                )
            )


            survival_times = (
                rsf.unique_times_
            )


            survival_probabilities = (
                np.array([
                    fn(
                        survival_times
                    )
                    for fn
                    in survival_functions
                ])
            )


            (
                metrics,
                calibration,
                predicted_rmst,
                predicted_median,
                median_reached
            ) = evaluate_model(
                "Random Survival Forest",
                event_train,
                duration_train,
                event_test,
                duration_test,
                risk,
                survival_probabilities,
                survival_times
            )


            metrics[
                "fold"
            ] = fold


            for row in calibration:

                row[
                    "fold"
                ] = fold


            calibration_results.extend(
                calibration
            )


            fold_results.append(
                metrics
            )


            store_predictions(
                "Random Survival Forest",
                risk,
                predicted_rmst,
                predicted_median,
                median_reached
            )


        except Exception as e:

            fold_results.append({
                "fold":
                    fold,
                "model":
                    "Random Survival Forest",
                "error":
                    str(e)
            })


        # =====================================================
        # 5. GRADIENT BOOSTING SURVIVAL
        # =====================================================

        try:

            gb = (
                GradientBoostingSurvivalAnalysis(
                    loss="coxph",
                    n_estimators=
                        gb_estimators,
                    learning_rate=
                        0.05,
                    max_depth=
                        3,
                    random_state=
                        random_state
                )
            )


            gb.fit(
                X_train_imputed,
                y_train
            )


            risk = (
                gb.predict(
                    X_test_imputed
                )
            )


            survival_functions = (
                gb
                .predict_survival_function(
                    X_test_imputed
                )
            )


            survival_times = (
                np.unique(
                    duration_train[
                        event_train
                    ]
                )
            )


            survival_probabilities = (
                np.array([
                    fn(
                        survival_times
                    )
                    for fn
                    in survival_functions
                ])
            )


            (
                metrics,
                calibration,
                predicted_rmst,
                predicted_median,
                median_reached
            ) = evaluate_model(
                "Gradient Boosting Survival",
                event_train,
                duration_train,
                event_test,
                duration_test,
                risk,
                survival_probabilities,
                survival_times
            )


            metrics[
                "fold"
            ] = fold


            for row in calibration:

                row[
                    "fold"
                ] = fold


            calibration_results.extend(
                calibration
            )


            fold_results.append(
                metrics
            )


            store_predictions(
                "Gradient Boosting Survival",
                risk,
                predicted_rmst,
                predicted_median,
                median_reached
            )


        except Exception as e:

            fold_results.append({
                "fold":
                    fold,
                "model":
                    "Gradient Boosting Survival",
                "error":
                    str(e)
            })


        # =====================================================
        # 6. SURVIVAL SVM
        # =====================================================

        try:

            svm = (
                FastSurvivalSVM(
                    alpha=1.0,
                    rank_ratio=1.0,
                    max_iter=1000,
                    random_state=
                        random_state
                )
            )


            svm.fit(
                X_train_scaled,
                y_train
            )


            risk = (
                svm.predict(
                    X_test_scaled
                )
            )


            (
                metrics,
                calibration,
                _,
                _,
                _
            ) = evaluate_model(
                "Survival SVM",
                event_train,
                duration_train,
                event_test,
                duration_test,
                risk_scores=risk
            )


            metrics[
                "fold"
            ] = fold


            fold_results.append(
                metrics
            )


            store_predictions(
                "Survival SVM",
                risk=risk
            )


        except Exception as e:

            fold_results.append({
                "fold":
                    fold,
                "model":
                    "Survival SVM",
                "error":
                    str(e)
            })


        # =====================================================
        # 7. DEEPSURV
        # =====================================================

        try:

            np.random.seed(
                random_state
                + fold
            )

            torch.manual_seed(
                random_state
                + fold
            )


            x_train_deep = (
                X_train_scaled
                .astype(
                    "float32"
                )
            )

            x_test_deep = (
                X_test_scaled
                .astype(
                    "float32"
                )
            )


            duration_train_deep = (
                duration_train
                .astype(
                    "float32"
                )
            )

            event_train_deep = (
                event_train
                .astype(
                    "float32"
                )
            )


            rng = (
                np.random.RandomState(
                    random_state
                    + fold
                )
            )

            indices = (
                np.arange(
                    len(
                        x_train_deep
                    )
                )
            )

            rng.shuffle(
                indices
            )


            val_size = max(
                1,
                int(
                    len(indices)
                    * 0.20
                )
            )


            val_idx = (
                indices[
                    :val_size
                ]
            )

            deep_train_idx = (
                indices[
                    val_size:
                ]
            )


            x_deep_train = (
                x_train_deep[
                    deep_train_idx
                ]
            )

            x_deep_val = (
                x_train_deep[
                    val_idx
                ]
            )


            duration_deep_train = (
                duration_train_deep[
                    deep_train_idx
                ]
            )

            duration_deep_val = (
                duration_train_deep[
                    val_idx
                ]
            )


            event_deep_train = (
                event_train_deep[
                    deep_train_idx
                ]
            )

            event_deep_val = (
                event_train_deep[
                    val_idx
                ]
            )


            net = (
                tt.practical.MLPVanilla(
                    x_train_deep.shape[1],

                    list(
                        deepsurv_nodes
                    ),

                    1,

                    batch_norm=True,
                    dropout=0.1,
                    output_bias=False
                )
            )


            deepsurv = (
                DeepSurvCoxPH(
                    net,
                    tt.optim.Adam
                )
            )


            deepsurv.optimizer.set_lr(
                0.001
            )


            callbacks = [
                tt.callbacks.EarlyStopping(
                    patience=10
                )
            ]


            deepsurv.fit(
                x_deep_train,

                (
                    duration_deep_train,
                    event_deep_train
                ),

                batch_size=256,

                epochs=
                    deepsurv_epochs,

                callbacks=
                    callbacks,

                val_data=(
                    x_deep_val,

                    (
                        duration_deep_val,
                        event_deep_val
                    )
                ),

                verbose=False
            )


            deepsurv.compute_baseline_hazards()


            risk = (
                deepsurv
                .predict(
                    x_test_deep
                )
                .reshape(-1)
            )


            surv_df = (
                deepsurv
                .predict_surv_df(
                    x_test_deep
                )
            )


            survival_times = (
                surv_df
                .index
                .to_numpy(
                    dtype=float
                )
            )


            survival_probabilities = (
                surv_df
                .T
                .to_numpy()
            )


            (
                metrics,
                calibration,
                predicted_rmst,
                predicted_median,
                median_reached
            ) = evaluate_model(
                "DeepSurv",
                event_train,
                duration_train,
                event_test,
                duration_test,
                risk,
                survival_probabilities,
                survival_times
            )


            metrics[
                "fold"
            ] = fold


            for row in calibration:

                row[
                    "fold"
                ] = fold


            calibration_results.extend(
                calibration
            )


            fold_results.append(
                metrics
            )


            store_predictions(
                "DeepSurv",
                risk,
                predicted_rmst,
                predicted_median,
                median_reached
            )


        except Exception as e:

            fold_results.append({
                "fold":
                    fold,
                "model":
                    "DeepSurv",
                "error":
                    str(e)
            })


    # =========================================================
    # DATAFRAMES
    # =========================================================

    fold_results = pd.DataFrame(
        fold_results
    )

    calibration_results = pd.DataFrame(
        calibration_results
    )

    predictions_df = pd.DataFrame(
        prediction_results
    )


    # =========================================================
    # SUMMARY
    # =========================================================

    metric_columns = [
        "c_index",

        *[
            f"brier_{h}"
            for h in horizons
        ],

        "integrated_brier_score",

        *[
            f"auc_{h}"
            for h in horizons
        ],

        f"restricted_mae_{restricted_time_horizon}",
        f"restricted_rmse_{restricted_time_horizon}",

        *[
            f"calibration_{h}"
            for h in horizons
        ]
    ]


    summary_rows = []


    for (
        model_name,
        model_data
    ) in fold_results.groupby(
        "model"
    ):

        row = {
            "model":
                model_name
        }


        for metric in metric_columns:

            if (
                metric
                not in model_data.columns
            ):
                continue


            values = (
                model_data[
                    metric
                ]
                .dropna()
            )


            if len(values) == 0:

                row[
                    f"{metric}_mean"
                ] = np.nan

                row[
                    f"{metric}_std"
                ] = np.nan

            else:

                row[
                    f"{metric}_mean"
                ] = values.mean()

                row[
                    f"{metric}_std"
                ] = values.std()


        summary_rows.append(
            row
        )


    summary_results = (
        pd.DataFrame(
            summary_rows
        )
    )


    if (
        "c_index_mean"
        in summary_results.columns
    ):

        summary_results = (
            summary_results
            .sort_values(
                "c_index_mean",
                ascending=False,
                na_position="last"
            )
            .reset_index(
                drop=True
            )
        )


    # =========================================================
    # PRINT MEDIAN INFORMATION
    # =========================================================

    if verbose:

        print(
            "\n=============================================="
        )

        print(
            "CROSS-VALIDATION SUMMARY"
        )

        print(
            "=============================================="
        )

        print(
            summary_results
        )


        if (
            not predictions_df.empty
            and
            "median_reached"
            in predictions_df.columns
        ):

            median_stats = (
                predictions_df
                .dropna(
                    subset=[
                        "median_reached"
                    ]
                )
                .groupby(
                    "model"
                )[
                    "median_reached"
                ]
                .agg(
                    [
                        "mean",
                        "count"
                    ]
                )
            )

            median_stats[
                "median_reached_pct"
            ] = (
                median_stats[
                    "mean"
                ]
                * 100
            )


            print(
                "\n=============================================="
            )

            print(
                "MEDIAN SURVIVAL REACHED"
            )

            print(
                "=============================================="
            )

            print(
                median_stats[
                    [
                        "median_reached_pct",
                        "count"
                    ]
                ]
            )


    # =========================================================
    # RETURN
    # =========================================================

    return (
        fold_results,
        summary_results,
        calibration_results,
        predictions_df,
        feature_columns
    )

In [ ]:
(
    fold_results,
    model_summary,
    calibration_results,
    predictions_df,
    features_used
) = cross_validate_survival_models(
    temporal_df,
    duration_col="remainingDays",
    event_col="eventHasHappend",
    group_col="id",
    n_splits=5,
    horizons=(7, 30, 180, 365),
    restricted_time_horizon=365
)

In [ ]:
model_summary[
    [
        "model",
        "c_index_mean",
        "brier_365_mean",
        "integrated_brier_score_mean",
        "auc_7_mean",
        "auc_30_mean",
        "auc_180_mean",
        "auc_365_mean",
        "calibration_365_mean"
    ]
]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error


def plot_true_vs_predicted_survival(
    predictions_df,
    true_col="remainingDays",
    event_col="eventHasHappend",
    model_col="model",
    prediction_col="predicted_time",
    only_events=True,
    max_time=None,
    figsize=(12, 5),
    alpha=0.35,
    save_folder=None
):
    """
    Create two plots for every survival model:

    1. True vs. predicted leave time scatter plot
    2. QQ plot comparing distributions of true and predicted leave times

    Parameters
    ----------
    predictions_df : pd.DataFrame
        Expected columns:
            - true survival / remaining time
            - event indicator
            - model name
            - predicted survival time

    true_col : str
        Column containing the true remaining survival time.

    event_col : str
        Event indicator:
            True / 1 = leave event observed
            False / 0 = censored

    model_col : str
        Column identifying the model.

    prediction_col : str
        Predicted remaining survival time.

    only_events : bool
        If True, only observations with observed events are evaluated.
        Recommended for true-vs-predicted plots.

    max_time : float or None
        Optional maximum time shown/evaluated.
        Example: 365.

    figsize : tuple
        Figure size.

    alpha : float
        Transparency of scatter points.

    save_folder : str or None
        If supplied, plots are saved as PNG files.

    Returns
    -------
    pd.DataFrame
        Summary containing:
            model
            n
            MAE
            RMSE
            Pearson correlation
            Spearman correlation
    """

    import os

    data = predictions_df.copy()

    # =========================================================
    # Prepare folder
    # =========================================================

    if save_folder is not None:

        os.makedirs(
            save_folder,
            exist_ok=True
        )

    # =========================================================
    # Numeric conversion
    # =========================================================

    data[true_col] = pd.to_numeric(
        data[true_col],
        errors="coerce"
    )

    data[prediction_col] = pd.to_numeric(
        data[prediction_col],
        errors="coerce"
    )

    # =========================================================
    # Only observed events
    # =========================================================

    if only_events:

        data = data[
            data[event_col].astype(bool)
        ].copy()

    # =========================================================
    # Remove missing values
    # =========================================================

    data = data.dropna(
        subset=[
            true_col,
            prediction_col,
            model_col
        ]
    )

    # =========================================================
    # Optional restriction
    # =========================================================

    if max_time is not None:

        data = data[
            data[true_col] <= max_time
        ].copy()

        data[prediction_col] = (
            data[prediction_col]
            .clip(
                lower=0,
                upper=max_time
            )
        )

    # =========================================================
    # Results
    # =========================================================

    results = []

    # =========================================================
    # Loop through models
    # =========================================================

    for model_name in data[model_col].unique():

        model_data = data[
            data[model_col] == model_name
        ].copy()

        true_values = (
            model_data[true_col]
            .to_numpy()
        )

        predicted_values = (
            model_data[prediction_col]
            .to_numpy()
        )

        if len(model_data) < 2:
            continue

        # =====================================================
        # Statistics
        # =====================================================

        mae = np.mean(
            np.abs(
                true_values
                - predicted_values
            )
        )

        rmse = np.sqrt(
            np.mean(
                (
                    true_values
                    - predicted_values
                ) ** 2
            )
        )

        try:

            pearson_r = stats.pearsonr(
                true_values,
                predicted_values
            ).statistic

        except Exception:

            pearson_r = np.nan

        try:

            spearman_r = stats.spearmanr(
                true_values,
                predicted_values
            ).statistic

        except Exception:

            spearman_r = np.nan

        results.append({
            "model": model_name,
            "n": len(model_data),
            "mae": mae,
            "rmse": rmse,
            "pearson_r": pearson_r,
            "spearman_r": spearman_r
        })

        # =====================================================
        # CREATE FIGURE
        # =====================================================

        fig, axes = plt.subplots(
            1,
            2,
            figsize=figsize
        )

        # =====================================================
        # LEFT: TRUE VS PREDICTED
        # =====================================================

        axes[0].scatter(
            true_values,
            predicted_values,
            alpha=alpha
        )

        # Determine common limits
        minimum = min(
            true_values.min(),
            predicted_values.min()
        )

        maximum = max(
            true_values.max(),
            predicted_values.max()
        )

        if max_time is not None:

            minimum = 0
            maximum = max_time

        # Perfect prediction line
        axes[0].plot(
            [minimum, maximum],
            [minimum, maximum],
            linestyle="--",
            label="Perfect prediction"
        )

        axes[0].set_xlim(
            minimum,
            maximum
        )

        axes[0].set_ylim(
            minimum,
            maximum
        )

        axes[0].set_xlabel(
            "True Leave Time (Days)"
        )

        axes[0].set_ylabel(
            "Predicted Leave Time (Days)"
        )

        axes[0].set_title(
            "True vs. Predicted Leave Time"
        )

        axes[0].legend()

        axes[0].grid(
            alpha=0.2
        )

        # =====================================================
        # RIGHT: QQ PLOT
        # =====================================================
        #
        # This is a two-sample QQ plot:
        #
        # quantiles of true leave times
        # vs.
        # quantiles of predicted leave times
        #
        # Points on y=x indicate similar distributions.
        # =====================================================

        quantiles = np.linspace(
            0.01,
            0.99,
            100
        )

        true_quantiles = np.quantile(
            true_values,
            quantiles
        )

        predicted_quantiles = np.quantile(
            predicted_values,
            quantiles
        )

        axes[1].scatter(
            true_quantiles,
            predicted_quantiles,
            alpha=0.7
        )

        qq_min = min(
            true_quantiles.min(),
            predicted_quantiles.min()
        )

        qq_max = max(
            true_quantiles.max(),
            predicted_quantiles.max()
        )

        if max_time is not None:

            qq_min = 0
            qq_max = max_time

        # Perfect distribution line
        axes[1].plot(
            [qq_min, qq_max],
            [qq_min, qq_max],
            linestyle="--",
            label="Perfect agreement"
        )

        axes[1].set_xlim(
            qq_min,
            qq_max
        )

        axes[1].set_ylim(
            qq_min,
            qq_max
        )

        axes[1].set_xlabel(
            "True Leave Time Quantiles (Days)"
        )

        axes[1].set_ylabel(
            "Predicted Leave Time Quantiles (Days)"
        )

        axes[1].set_title(
            "QQ Plot"
        )

        axes[1].legend()

        axes[1].grid(
            alpha=0.2
        )

        # =====================================================
        # MAIN TITLE
        # =====================================================

        fig.suptitle(
            f"{model_name}\n"
            f"MAE={mae:.1f} days | "
            f"RMSE={rmse:.1f} days | "
            f"Pearson r={pearson_r:.2f} | "
            f"Spearman ρ={spearman_r:.2f}",
            fontsize=13
        )

        plt.tight_layout()

        # =====================================================
        # SAVE
        # =====================================================

        if save_folder is not None:

            safe_name = (
                str(model_name)
                .replace(" ", "_")
                .replace("/", "_")
            )

            filepath = os.path.join(
                save_folder,
                f"{safe_name}_true_vs_predicted.png"
            )

            plt.savefig(
                filepath,
                dpi=300,
                bbox_inches="tight"
            )

        plt.show()

    # =========================================================
    # RETURN SUMMARY
    # =========================================================

    results = pd.DataFrame(
        results
    )

    if not results.empty:

        results = (
            results
            .sort_values("mae")
            .reset_index(drop=True)
        )

    return results

In [ ]:
predictions_df[
    [
        "id",
        "model",
        "remainingDays",
        "eventHasHappend",
        "predicted_rmst",
        "predicted_median_time",
        "median_reached"
    ]
]

In [ ]:
prediction_summary = plot_true_vs_predicted_survival(
    predictions_df,
    true_col="remainingDays",
    event_col="eventHasHappend",
    model_col="model",
    prediction_col="predicted_median_time",
    only_events=True,
    max_time=365*2
)